<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Prompt Engineering für Agenten
</b></font> </br></p>

---

In [2]:
#@title  🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "M04-Prompt-Engineering"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

✓ OPENAI_API_KEY erfolgreich gesetzt
✓ LANGSMITH_API_KEY erfolgreich gesetzt

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.12
langchain-chroma                         1.1.0
langchain-classic                        1.0.3
langchain-community                      0.4.1
langchain-core                           1.2.19
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.1.2
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.11

IP-Adresse: 34.125.80.120
Hostname: 120.80.125.34.bc.googleusercontent.com
Stadt: Las Vegas
Region: Nevada
Land: US
Koordinaten: 36.1750,-115.1372
Provider: AS396982 Google LLC
Postleitzahl: 89111
Zeitzone: America/Los_Angeles


ModuleNotFoundError: No module named 'genai_lib.model_config'

# 1 | Übersicht
---


Prompts sind die einzige Schnittstelle zwischen dem Entwickler und dem Modell — was hier nicht steht, bleibt dem Zufall überlassen.


In [ ]:
# Imports für das Modul
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model(BASELINE, temperature=0)

# 2 | ChatPromptTemplate
---

`ChatPromptTemplate` trennt klar zwischen Rollen (`system`, `human`, `ai`) und Variablen.
Das ist robuster als Prompt-Strings zusammenzukleben.


**`load_prompt()` – Prompts als versionierte Assets**

Statt Prompts direkt im Code zu schreiben, laedt `load_prompt()` ein Markdown-Dokument aus einem lokalen Pfad oder einer GitHub-URL und gibt ein fertiges `ChatPromptTemplate` zurueck.

```python
# Aus GitHub (Blob- und Tree-URLs werden automatisch in Raw-Links umgewandelt)
prompt = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_python_tutor_prompt.md",
    mode="T"   # "T" = ChatPromptTemplate | "S" = plain String
)

# Oder lokal:
prompt = load_prompt("../05_prompt/m04_python_tutor_prompt.md", mode="T")
```



**Prompt-Format (Markdown mit YAML-Frontmatter):**
```markdown
---
name: python_tutor
description: Erklaert Python-Konzepte fuer verschiedene Zielgruppen
variables: [thema, zielgruppe]
---

## system
Du bist ein Python-Tutor fuer {zielgruppe}.

## human
Erklaere {thema} mit einem Beispiel.
```



**Vorteile gegenüber Inline-Prompts:**

| | Inline im Code | `load_prompt()` aus Datei |
|---|---|---|
| **Versionierung** | Im Code-Commit versteckt | Eigener Commit, diff-bar |
| **Uebersichtlichkeit** | Code und Prompts vermischt | Klar getrennt |
| **Wiederverwendung** | Copy-Paste | Eine Datei, viele Notebooks |
| **Zusammenarbeit** | Erfordert Code-Kenntnisse | Einfaches Markdown |

> Das `05_prompt/`-Verzeichnis im Repository ist damit ein **Prompt Hub** – alle Prompts an einem Ort, versioniert per Git. Alle Notebooks dieses Kurses nutzen dieses Muster.

In [ ]:
prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_python_tutor_prompt.md", mode="T")

messages = prompt.invoke({
    "thema": "for-Schleifen",
    "zielgruppe": "Einsteiger"
})

for msg in messages.messages:
    print(f"{msg.type.upper()}: {msg.content}")

In [ ]:
chain = prompt | llm
response = chain.invoke({"thema": "List Comprehension", "zielgruppe": "Data-Analysten"})
mprint(response.content)

# 3 | System Prompts für Tool-Calling
---



Für Agenten mit Tools sollte der `system`-Prompt explizit festlegen:
- wann ein Tool verwendet wird
- wann **kein** Tool verwendet wird
- welches Ausgabeformat erwartet wird


In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool

@tool
def add(a: float, b: float) -> float:
    """Addiert zwei Zahlen und gibt das Ergebnis zurueck."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multipliziert zwei Zahlen und gibt das Ergebnis zurueck."""
    return a * b

system_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_rechenassistent_system_prompt.md", mode="S")

agent = create_agent(
    model=BASELINE,
    tools=[add, multiply],
    system_prompt=system_prompt,
)

result = agent.invoke({"messages": [{"role": "user", "content": "(12 + 8) * 3"}]})
mprint(result["messages"][-1].content)


# 4 | Few-Shot Examples
---



Few-Shot-Beispiele helfen, Stil und Struktur der Antworten zu stabilisieren.
Nutzen Sie wenige, aber repräsentative Beispiele.


In [ ]:
zero_shot = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_ticket_zero_shot_prompt.md", mode="T")
few_shot = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m04_ticket_few_shot_prompt.md", mode="T")
for name, p in [("Zero-Shot", zero_shot), ("Few-Shot", few_shot)]:
    # 1. Tracing-Konfiguration vorab festlegen
    run_cfg = {
        "run_name": f"M04_Kap4_{name.replace('-', '')}",
        "tags": ["M04", "few-shot", name.lower().replace("-", "")]
    }
    # 2. with_config() anwenden, dann invoke()
    out = (p | llm).with_config(**run_cfg).invoke({"ticket": "Wo finde ich meine letzte Rechnung?"})
    print()
    print(f"{name}: {out.content.strip()}")


# 5 | Prompt-Varianten testen
---



Ein einfacher Vergleich hilft, bessere Prompts datenbasiert auszuwählen.
Unten testen wir mehrere System-Prompts auf dieselbe Anfrage.


In [ ]:
variants = {
    "knapp": "Antworte in maximal 2 Saetzen.",
    "didaktisch": "Erklaere schrittweise mit kurzem Beispiel.",
    "kritisch": "Zeige Antwort und nenne eine moegliche Fehlerquelle.",
}
from langchain_core.prompts import ChatPromptTemplate
variant_prompt = ChatPromptTemplate.from_messages([
    ("system", "Du bist Python-Coach. {instruction}"),
    ("human", "{q}"),
])
user_question = "Wann sollte ich in Python eine Liste statt eines Sets verwenden?"
for name, instruction in variants.items():
    # 1. Tracing-Konfiguration vorab festlegen
    run_cfg = {
        "run_name": f"M04_Kap5_Variant_{name}",
        "tags": ["M04", "prompt-variants", name]
    }
    # 2. with_config() anwenden, dann invoke()
    answer = (variant_prompt | llm).with_config(**run_cfg).invoke({"instruction": instruction, "q": user_question})
    mprint("### Variante: " + name)
    mprint('---')
    mprint(answer.content.strip())


# 6 | Strukturierte Agenten-Prompts: Production Pattern
---

LangGraph-Produktions-Prompts folgen einem bewährten Muster mit 5 Patterns,
die Agenten zuverlässiger und kontrollierbarer machen.

## Pattern 1: XML-Tags als Struktur-Trenner

LLMs folgen klar abgegrenzten Sektionen zuverlässiger als Fließtext.
Jede Sektion hat einen eigenen kognitiven Zweck.

```python
# ❌ Unstrukturiert – LLM muss Regeln aus Text extrahieren
system = "Du bist Assistent. Nutze Tools. Sei effizient. Stoppe wenn fertig."

# ✅ Strukturiert – klare Trennung von Rolle, Prozess und Grenzen
system = """
<Task>Was der Agent tun soll</Task>
<Instructions>Wie er vorgehen soll</Instructions>
<Hard Limits>Wann er stoppen soll</Hard Limits>
"""
```

Fehlt die XML-Struktur, übersieht das Modell häufig Regeln, die weiter unten im Prompt stehen — besonders bei langen System-Prompts mit mehr als 500 Tokens.

## Pattern 2: Explizite Tool-Steuerung

Ohne Anweisung nutzen Agenten Tools opportunistisch – springen direkt zur nächsten Aktion.
Explizite Reflexions-Aufforderung erzwingt Bewertung nach jedem Schritt.

```python
"KRITISCH: Bewerte nach jedem Tool-Ergebnis: Habe ich genug um zu antworten?"
```

## Pattern 3: Quantitative Limits statt qualitativer Anweisungen

```python
# ❌ Qualitativ – nicht handlungsleitend für ein LLM
"Sei sparsam mit Tool-Calls."

# ✅ Quantitativ – konkret und messbar
"Tool-Budget: maximal 3 Suchen. Nach 3 Suchen immer stoppen."
```

## Pattern 4: Explizite Stop-Bedingungen

Agenten neigen zu Over-Research. Ohne Stop-Kriterien laufen sie bis zum recursion_limit.

```python
"""Sofort stoppen wenn:
- Du die Frage vollständig beantworten kannst
- Die letzte Suche keine neuen Informationen brachte
- Du 3+ relevante Quellen gefunden hast
"""
```

## Pattern 5: Strategie vorgeben (Breit → Eng)

Ein mentales Modell der Vorgehensweise ist wirkungsvoller als ein Regelwerk.

```python
"""<Instructions>
1. Lies die Anfrage sorgfältig
2. Starte mit breiteren Suchanfragen (allgemeines Thema zuerst)
3. Verfeinere mit spezifischeren Suchen wenn Infos fehlen
4. Stoppe sobald du die Anfrage beantworten kannst
</Instructions>
"""
```

**Beispiel unten:** Alle 5 Patterns in einem praktischen Agenten-Prompt.

In [ ]:
# Strukturierter System-Prompt mit allen 5 Patterns
from langchain_core.tools import tool

structured_system_prompt = """Du bist ein Python-Tutor fuer den KI-Agenten-Kurs.

<Task>
Beantworte Python-Fragen klar, praxisnah und mit Beispielen.
</Task>

<Instructions>
1. Lies die Frage sorgfaeltig – Grundlagen oder fortgeschrittenes Thema?
2. Beginne mit dem Kernkonzept, dann ein konkretes Beispiel
3. Pruefe: Ist die Erklaerung fuer die Zielgruppe verstaendlich?
4. Ergaenze einen Praxistipp wenn hilfreich
</Instructions>

<Hard Limits>
Maximale Antwortlaenge: 150 Woerter.
Sofort antworten wenn: Kernkonzept und Beispiel erklaert sind.
</Hard Limits>"""

@tool
def python_beispiel(konzept: str) -> str:
    """Gibt ein konkretes Python-Code-Beispiel fuer ein Konzept zurueck."""
    beispiele = {
        "list comprehension": "[x**2 for x in range(10) if x % 2 == 0]",
        "lambda":             "quadrat = lambda x: x ** 2",
        "decorator":          "@functools.wraps(func)",
        "generator":          "def zahlen(): yield from range(10)",
    }
    return beispiele.get(konzept.lower(), f"# Beispiel fuer: {konzept}\npass")

agent = create_agent(
    model=BASELINE,
    tools=[python_beispiel],
    system_prompt=structured_system_prompt,
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Erkläre List Comprehensions fuer Einsteiger."}]
})
mprint(result["messages"][-1].content)

# 7 | LangSmith: Prompt-Inspektion
---



LangSmith macht Prompt-Experimente nachvollziehbar: Welcher Prompt erzeugte welche Antwort?

**Empfehlung im Kursbetrieb:**
- `run_name` pro Experiment setzen
- `tags` für Modul/Kapitel verwenden
- bei Vergleichen immer denselben Input nutzen

**Hinweis (EU):** Endpoint `https://eu.api.smith.langchain.com` – bereits in der Setup-Cell konfiguriert.

In [ ]:
# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "M04_Kap6_PromptBaseline",
    "tags":     ["M04", "prompt-engineering"]
}

# 2. with_config() anwenden, dann invoke()
traced_chain = (prompt | llm).with_config(**run_cfg)
traced_response = traced_chain.invoke({
    "thema": "while-Schleifen",
    "zielgruppe": "Python-Einsteiger"
})

mprint("### Trace erstellt")
mprint('---')
mprint(traced_response.content)

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M04-Prompt-Engineering", limit=3, show_steps=True)

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, es kann aber auch gerne eine andere Herausforderung angegangen werden.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Wenn bei einer Aufgabe eine Blockade entsteht, kann zum Beispiel Gemini in Google Colab verwendet werden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

**Grundlagen**
- Einen klaren System-Prompt für einen einfachen Agenten schreiben
- Mit 1-2 Beispielanfragen testen

In [ ]:
# ✅ Selbstcheck Grundlagen
# Voraussetzung: System-Prompt in 'mein_system_prompt' (String) gespeichert,
# Agent mit diesem Prompt in 'mein_agent' definiert und mindestens 1 Testanfrage ausgeführt.

assert 'mein_system_prompt' in dir() or 'mein_system_prompt' in locals(), \
    "❌ Variable 'mein_system_prompt' fehlt – speichern Sie Ihren System-Prompt als String"
assert isinstance(mein_system_prompt, str) and len(mein_system_prompt) >= 20, \
    "❌ System-Prompt zu kurz – mindestens 20 Zeichen erwartet"
assert 'mein_agent' in dir() or 'mein_agent' in locals(), \
    "❌ Variable 'mein_agent' fehlt – erstellen Sie einen Agenten und speichern Sie ihn in 'mein_agent'"
assert hasattr(mein_agent, "invoke"), \
    "❌ 'mein_agent' hat kein invoke() – wurde der Agent mit create_agent() erstellt?"
print("✅ Grundlagen-Selbstcheck bestanden!")

**Aufbau**
1. Einen Agenten mit 2-3 Tools erstellen (z. B. Rechnen, Datum, String-Utils).
2. Zwei unterschiedliche System-Prompts schreiben: einmal „knapp", einmal „didaktisch".
3. 3 Testfragen definieren und beide Varianten vergleichen.
4. Kurz dokumentieren: Welche Prompt-Version ist für den gewählten Use Case besser und warum?

In [ ]:
# ✅ Selbstcheck Aufbau
# Voraussetzung: Prompt-Varianten in 'meine_varianten' als Dict gespeichert
# (mindestens 2 Einträge, z.B. {"knapp": "...", "didaktisch": "..."}).
# Agent mit 2-3 Tools in 'mein_agent' definiert.

assert 'meine_varianten' in dir() or 'meine_varianten' in locals(), \
    "❌ Variable 'meine_varianten' fehlt – speichern Sie die Prompt-Varianten als Dict"
assert isinstance(meine_varianten, dict), \
    "❌ 'meine_varianten' muss ein Dict sein (z.B. {'knapp': '...', 'didaktisch': '...'})"
assert len(meine_varianten) >= 2, \
    f"❌ Nur {len(meine_varianten)} Variante(n) – mindestens 2 unterschiedliche Prompt-Varianten erwartet"
assert all(isinstance(v, str) and len(v) >= 10 for v in meine_varianten.values()), \
    "❌ Jede Variante muss ein String mit mindestens 10 Zeichen sein"

# Tools prüfen
assert 'meine_tools' in dir() or 'meine_tools' in locals(), \
    "❌ Variable 'meine_tools' fehlt – speichern Sie Ihre Tools als Liste in 'meine_tools'"
assert isinstance(meine_tools, list) and len(meine_tools) >= 2, \
    f"❌ Nur {len(meine_tools) if isinstance(meine_tools, list) else 0} Tool(s) – mindestens 2 Tools für den Aufbau erwartet"
print(f"✅ Aufbau-Selbstcheck bestanden! ({len(meine_varianten)} Varianten, {len(meine_tools)} Tools)")

**Vertiefung**
- Ein Few-Shot-Beispiel in einer der beiden Prompt-Varianten ergänzen
- Eine klare Regel formulieren, wann der Agent Tools vermeiden soll

In [ ]:
# ✅ Selbstcheck Vertiefung
# Voraussetzung: Few-Shot-Beispiel in 'mein_few_shot_prompt' (ChatPromptTemplate oder String),
# Dokumentation der Tool-Vermeidungsregel in 'tool_vermeidungsregel' (String).

import os

assert os.environ.get("LANGSMITH_TRACING") == "true", \
    "❌ LangSmith-Tracing nicht aktiv – LANGSMITH_TRACING muss 'true' sein (siehe Setup-Zelle)"

assert 'mein_few_shot_prompt' in dir() or 'mein_few_shot_prompt' in locals(), \
    "❌ Variable 'mein_few_shot_prompt' fehlt – ergänzen Sie ein Few-Shot-Beispiel in Ihrem Prompt"

assert 'tool_vermeidungsregel' in dir() or 'tool_vermeidungsregel' in locals(), \
    "❌ Variable 'tool_vermeidungsregel' fehlt – formulieren Sie eine Regel wann der Agent keine Tools nutzen soll"
assert isinstance(tool_vermeidungsregel, str) and len(tool_vermeidungsregel) >= 20, \
    "❌ Regel zu kurz – mindestens 20 Zeichen für eine sinnvolle Tool-Vermeidungsregel erwartet"
print("✅ Vertiefung-Selbstcheck bestanden!")